In [69]:
%pip install dmba


In [70]:
# Imports pandas for loading and manipulating tabular data.
import pandas as pd
# Imports the decision-tree classifier used when the dependent variable is categorical or binary.
from sklearn.tree import DecisionTreeClassifier #if your dependent variable is binary
# Imports the decision-tree regressor used when the dependent variable is numerical.
from sklearn.tree import DecisionTreeRegressor  #if your dependent variable is numerical, like Sales
# Imports the random-forest classifier, which combines many decision trees for classification.
from sklearn.ensemble import RandomForestClassifier
# Imports StandardScaler for standardizing numerical predictor variables.
from sklearn.preprocessing import StandardScaler
# Imports train_test_split for dividing data into training and testing sets.
from sklearn.model_selection import train_test_split
# Imports the K-nearest neighbors classifier.
from sklearn.neighbors import KNeighborsClassifier
# Imports accuracy_score for calculating the proportion of correct predictions.
from sklearn.metrics import accuracy_score

# Imports classificationSummary for displaying classification performance results.
from dmba import classificationSummary
# Imports dmba helper functions for plotting decision trees and summarizing classification and regression results.
from dmba import plotDecisionTree, classificationSummary, regressionSummary

In [71]:
# Reads the diabetes.csv file into a pandas DataFrame named diabetes_df.
diabetes_df = pd.read_csv('diabetes.csv')
# Displays the first five rows of diabetes_df so you can inspect the data.
diabetes_df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [72]:
# Creates X by removing Outcome, leaving only the predictor variables used to make predictions.
X = diabetes_df.drop(columns=['Outcome'])
# Stores the Outcome column in y as the target variable the model will predict.
y = diabetes_df['Outcome']


In [73]:
# Splits X and y into training and testing sets, using 25% of the observations for testing and a fixed random seed for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)


In [74]:
# Initialize the StandardScaler
# Creates a StandardScaler object that will be used to standardize the predictor variables.
scaler = StandardScaler()

In [75]:
# Scale training set
# Learns the scaling parameters from X_train and uses them to standardize the training predictors.
X_train_scaled = scaler.fit_transform(X_train)
# Converts the scaled training data from a NumPy array back into a pandas DataFrame.
X_train_scaled = pd.DataFrame(X_train_scaled)

In [76]:
# Scale test set
# Uses the scaling parameters learned from the training data to standardize X_test.
X_test_scaled = scaler.transform(X_test)
# Converts the scaled test data from a NumPy array back into a pandas DataFrame.
# NOTICE: Scaled only the independent variable
X_test_scaled = pd.DataFrame(X_test_scaled)

### KNN model

In [77]:
# Creates a K-nearest neighbors classifier that uses the three nearest observations to classify each case.
# n_neighbors IS K IN KNN
knn = KNeighborsClassifier(n_neighbors=3)

In [78]:
# Trains the K-nearest neighbors classifier using the scaled training predictors and their known target values.
knn.fit(X_train_scaled, y_train)

KNeighborsClassifier(n_neighbors=3)

In [79]:
# Predicts the test-set classes with the trained KNN model and displays a classification performance summary against the actual test labels.
classificationSummary(y_test, knn.predict(X_test_scaled))

Confusion Matrix (Accuracy 0.7344)

       Prediction
Actual   0   1
     0 104  19
     1  32  37


In [80]:
# Train a classifier for different values of k
# Creates an empty list that will store the k value and accuracy from each model.
results = []
# Repeats the following indented steps for k values from 1 through 14.
for k in range(1, 15):
    # Creates and immediately trains a KNN classifier using the current value of k.
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train)
    # Adds a dictionary containing the current k value and its test accuracy to the results list.
    results.append({
        # Stores the current number of neighbors under the column name k.
        'k': k,
        # Predicts the test classes and calculates the accuracy for the current KNN model.
        'accuracy': accuracy_score(y_test, knn.predict(X_test_scaled))
    # Closes the dictionary and the append operation.
    })

# Convert results to a pandas data frame
# Converts the list of result dictionaries into a pandas DataFrame for easier comparison.
results = pd.DataFrame(results)
# Displays the table of k values and their corresponding test accuracies. HIGHER ACCURACY IS BEST K
print(results)

     k  accuracy
0    1  0.697917
1    2  0.692708
2    3  0.734375
3    4  0.729167
4    5  0.791667
5    6  0.781250
6    7  0.786458
7    8  0.786458
8    9  0.786458
9   10  0.776042
10  11  0.760417
11  12  0.786458
12  13  0.791667
13  14  0.786458


# Neural Network

In [81]:
# Imports the multilayer perceptron classifier used to build a neural network model.
from sklearn.neural_network import MLPClassifier

In [82]:
# Initialize the StandardScaler
# Creates a new StandardScaler object for standardizing the neural-network predictor variables.
scaler = StandardScaler()
# Scale training data
# Learns the scaling parameters from X_train and uses them to standardize the neural-network training predictors.
X_train_scaled = scaler.fit_transform(X_train)
# Converts the scaled training data from a NumPy array back into a pandas DataFrame.
X_train_scaled = pd.DataFrame(X_train_scaled)

In [83]:
# Finds the unique target-class codes in y, sorts them, and stores the ordered classes in a list.
classes = sorted(y.unique())
# Displays the class codes that will be used as labels in the classification summary.
print(classes)

[np.int64(0), np.int64(1)]


In [84]:
# Creates a neural network classifier with one hidden layer containing one neuron, logistic activation, and a fixed random seed.
clf = MLPClassifier(hidden_layer_sizes=[1], activation='logistic',  random_state=1) #one hidden layer with one neuron
# example, if you want to model two hiddle layers with two neurons you should change hidden_layer_sizes to: hidden_layer_sizes=(2, 2)
# Trains the neural network using the scaled training predictors and their known target values.
clf.fit(X_train_scaled, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(activation='logistic', hidden_layer_sizes=[1], random_state=1)

In [85]:
# Predicts the test-set classes with the trained neural network and displays a classification performance summary using the sorted class labels.
classificationSummary(y_test, clf.predict(X_test_scaled), class_names=classes)

Confusion Matrix (Accuracy 0.6406)

       Prediction
Actual   0   1
     0 123   0
     1  69   0


## Logistic regression

In [86]:
# Imports the LogisticRegression model from scikit-learn for binary classification. R-squared of only for linear regression
from sklearn.linear_model import LogisticRegression

In [87]:
# Creates a logistic regression model and stores it in the variable logit_reg.
logit_reg = LogisticRegression()
# Trains the logistic regression model using the training predictors and training outcome values.
logit_reg.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [88]:
# Evaluates the model on the training data by comparing actual outcomes with predicted outcomes. (predicted value)
classificationSummary(y_train, logit_reg.predict(X_train))

Confusion Matrix (Accuracy 0.7708)

       Prediction
Actual   0   1
     0 330  47
     1  85 114


In [89]:
# Evaluates the model on the testing data by comparing actual outcomes with predicted outcomes. (predicted value)
classificationSummary(y_test, logit_reg.predict(X_test))

Confusion Matrix (Accuracy 0.7760)

       Prediction
Actual   0   1
     0 109  14
     1  29  40


## Decision tree

In [90]:
# Creates an unrestricted decision-tree classifier with default settings.
fullClassTree = DecisionTreeClassifier(random_state=0)
# Trains the full decision tree on the training predictors and training target.
fullClassTree.fit(X_train, y_train)

DecisionTreeClassifier(random_state=0)

In [91]:
# Evaluates the full decision tree on the training data by comparing actual and predicted classes.
classificationSummary(y_train, fullClassTree.predict(X_train))

Confusion Matrix (Accuracy 1.0000)

       Prediction
Actual   0   1
     0 377   0
     1   0 199


In [92]:
# Evaluates the full decision tree on the test data by comparing actual and predicted classes.
classificationSummary(y_test, fullClassTree.predict(X_test))

Confusion Matrix (Accuracy 0.7031)

       Prediction
Actual  0  1
     0 98 25
     1 32 37


## Random Forest

In [93]:
# Creates a random-forest classifier containing 500 decision trees and fixes the random seed.
# random_state should always be 1 to give you the same results
rf = RandomForestClassifier(n_estimators=500, random_state=1)
# Trains the random-forest classifier on the training data.
rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=500, random_state=1)

In [94]:
# Evaluates the random forest on the test data by comparing actual and predicted classes.
classificationSummary(y_test, rf.predict(X_test))

Confusion Matrix (Accuracy 0.8177)

       Prediction
Actual   0   1
     0 110  13
     1  22  47
